In [ ]:
folder = 'exp_23Jul14-1530'

In [ ]:
# load
from collections import defaultdict
from tqdm import tqdm
import cloudpickle
import pathlib

folder = pathlib.Path(folder)
assert folder.exists()

def tree():
    return defaultdict(tree)


pval_seed_method_ana_effect = tree()

for file in tqdm(folder.glob('analysis*.p'), desc='load per experiment'):
    with open(file, 'rb') as f:
        pval, seed, method, ana, effect = cloudpickle.load(file=f)
        pval_seed_method_ana_effect[pval][seed][method] = ana, effect

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, confusion_matrix


# extract
pval_list = sorted(pval_seed_method_ana_effect.keys())
max_seed = max(max(pval_seed_method_ana_effect[p].keys()) for p in pval_list)

shape = max_seed + 1, len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for pval, t0 in pval_seed_method_ana_effect.items():
    for seed, t1 in t0.items():
        for method, (ana, effect) in t1.items():
            # build mask of predicted area (union of all effect masks)
            mask_pred = np.zeros(ana.exp.mask_idx.shape, dtype=bool)
            for _effect in ana.effect_tup:
                mask_pred |= _effect.mask

            # build y_true / y_pred in sklearn format
            mask_active = ana.exp.mask_idx > -1
            y_true = effect.mask[mask_active]
            y_pred = mask_pred[mask_active]
            
            # compute scores
            _f1 = f1_score(y_true=y_true, y_pred=y_pred, zero_division=0)
            _sens = recall_score(y_true=y_true, y_pred=y_pred, zero_division=0)
            conf_mat = confusion_matrix(y_true=y_true, y_pred=y_pred)

            # store
            pval_idx = pval_list.index(pval)
            score_dict[method, 'f1'][seed, pval_idx] = _f1
            score_dict[method, 'sens'][seed, pval_idx] = _sens
            score_dict[method, 'spec'][seed, pval_idx] = \
                conf_mat[0, 0] / (conf_mat[0, 0] + conf_mat[1, 0])

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(3, 2)
style_dict = {'AnalysisTFCE': '--', 'AnalysisHRBA': '-'}
for _ax, feat in zip(ax[:, 0], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, style in style_dict.items():
        line = plt.plot(pval_list, score_dict[method, feat].T, linestyle=style, color='k', label='_nolegend_')

    plt.xlabel('pval')
    plt.ylabel(feat)
    plt.xscale('log')
   
for method, style in style_dict.items():
    plt.plot([], [], linestyle=style, color='k', label=method[-4:])
plt.legend()

for _ax, feat in zip(ax[:, 1], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHRBA', feat] - score_dict['AnalysisTFCE', feat]
    plt.axhline([0], linewidth=2, color='k', alpha=.7)
    plt.plot(pval_list, x.T, linestyle=style, color='b')

    plt.xlabel('pval')
    plt.ylabel(f'{feat}: HRBA - TFCE')
    plt.xscale('log')
    
fig.set_size_inches(8, 8)
fig.tight_layout()

# Examine Worst Cases
Where HRBA does poorest as compared to TFCE

In [ ]:
import hrba.plot

%matplotlib tk
diff = score_dict['AnalysisHRBA', 'f1'] - score_dict['AnalysisTFCE', 'f1']

for idx in np.argsort(diff.flatten())[:1]:
    # lookup analysis & effect
    seed, pval_idx = np.unravel_index(idx, diff.shape)
    pval = pval_list[pval_idx]
    ana, effect = pval_seed_method_ana_effect[pval][seed]['AnalysisHRBA']

    # lookup / print f1 scores
    f1_hrba = score_dict['AnalysisHRBA', 'f1'][seed, pval_idx]
    f1_tfce = score_dict['AnalysisTFCE', 'f1'][seed, pval_idx]
    print(f'\n\nseed {seed} pval: {pval:.2E} HRBA f1: {f1_hrba:.3f} TFCE f1: {f1_tfce:.3f}')
    
    # plot
    epoch = ana.epoch_list[0]
    fig = hrba.plot.scatter_summary(epoch=epoch, mask=effect.mask)
    fig.set_size_inches(8, 15)
    plt.tight_layout()
    plt.show()

In [ ]:
ana.epoch_list[0].__dict__.keys()